%md
## Part 1: SQL path

Generates and executes SQL against the Gold layer to answer 
data/metric questions (e.g. "which state had the highest late 
delivery rate").

In [0]:
%run ./setup/02_service_principal_auth

In [0]:
%pip install langgraph
dbutils.library.restartPython()

In [0]:
%run ./setup/02_service_principal_auth

In [0]:
GOLD_SCHEMA_CONTEXT = """
Available tables (read-only, schema: gold):

1. gold.seller_performance
   - seller_id, seller_city, seller_state
   - total_orders, avg_review_score, avg_delivery_days, total_revenue

2. gold.sales_by_category_month
   - category, month, total_orders, total_revenue, avg_item_price

3. gold.delivery_delay_analysis
   - order_id, customer_state, order_purchase_timestamp
   - order_estimated_delivery_date, order_delivered_customer_date
   - delay_days, delivery_status ('late' or 'on_time')
"""

In [0]:
from openai import OpenAI

chat_client = OpenAI(
    api_key=sp_token,
    base_url=f"{WORKSPACE_URL}/ai-gateway/mlflow/v1"
)

embed_client = OpenAI(
    api_key=sp_token,
    base_url=f"{WORKSPACE_URL}/ai-gateway/mlflow/v1"
)


def generate_sql(question: str) -> str:
    response = chat_client.chat.completions.create(
        model="system.ai.gpt-oss-120b",
        messages=[
            {"role": "system", "content": f"""You are a SQL generator. Given the schema below, write a single read-only PostgreSQL SELECT query that answers the user's question. Return ONLY the SQL query, no explanation, no markdown formatting.

{GOLD_SCHEMA_CONTEXT}"""},
            {"role": "user", "content": question}
        ],
        max_tokens=1024
    )
    
    content = response.choices[0].message.content
    
    if isinstance(content, list):
        text = "".join(
            block.get("text", "") for block in content if isinstance(block, dict) and block.get("type") == "text"
        )
    else:
        text = content
    
    return text.strip()



def validate_sql(sql: str) -> bool:
    normalized = sql.strip().upper()
    forbidden = ["INSERT", "UPDATE", "DELETE", "DROP", "ALTER", "TRUNCATE", "CREATE", "GRANT"]
    if not normalized.startswith("SELECT"):
        return False
    if any(word in normalized for word in forbidden):
        return False
    return True

In [0]:
question = "Which state had the highest late delivery rate?"
sql = generate_sql(question)
print(sql)
print("Valid:", validate_sql(sql))

In [0]:
import time

def execute_gold_query(sql: str, warehouse_id: str) -> list:
    headers = {"Authorization": f"Bearer {sp_token}", "Content-Type": "application/json"}
    
    submit_response = requests.post(
        f"{WORKSPACE_URL}/api/2.0/sql/statements",
        headers=headers,
        json={
            "statement": sql,
            "warehouse_id": warehouse_id,
            "catalog": "olist_lakehouse",
            "schema": "gold"
        }
    )
    result = submit_response.json()
    statement_id = result["statement_id"]
    
    while result["status"]["state"] in ("PENDING", "RUNNING"):
        time.sleep(1)
        result = requests.get(
            f"{WORKSPACE_URL}/api/2.0/sql/statements/{statement_id}",
            headers=headers
        ).json()
    
    if result["status"]["state"] != "SUCCEEDED":
        raise Exception(f"Query failed: {result['status']}")
    
    return result["result"]["data_array"]

In [0]:
WAREHOUSE_ID = "d45bde8408adc9ed"

question = "Which state had the highest late delivery rate?"
sql = generate_sql(question)
print(f"Generated SQL:\n{sql}\n")

if validate_sql(sql):
    rows = execute_gold_query(sql, WAREHOUSE_ID)
    print(f"Result: {rows}")
else:
    print("Query rejected by security validation")

%md
## Debugging note: token expiration

While testing, the SQL execution call failed with a 403 "Invalid Token" 
error — the OAuth token generated via client-credentials has a short 
lifespan (appears to expire within a few minutes), and had expired 
during the warehouse's cold-start delay. This cell was used to inspect 
the raw API response and confirm the cause.

In [0]:
WAREHOUSE_ID = "d45bde8408adc9ed"

headers = {"Authorization": f"Bearer {sp_token}", "Content-Type": "application/json"}

submit_response = requests.post(
    f"{WORKSPACE_URL}/api/2.0/sql/statements",
    headers=headers,
    json={
        "statement": sql,
        "warehouse_id": WAREHOUSE_ID,
        "catalog": "olist_lakehouse",
        "schema": "gold"
    }
)

print("Status code:", submit_response.status_code)
print("Raw response:", submit_response.text)

%md
## Part 2: Vector search path

Embeds the question and searches the knowledge_base for semantically 
similar project documentation (e.g. "why did we get duplicate reviews").

In [0]:
def vector_search(question: str, top_k: int = 3) -> list:
    embedding_response = embed_client.embeddings.create(
        model="system.ai.qwen3-embedding-0-6b",
        input=question
    )
    question_embedding = embedding_response.data[0].embedding
    
    API_URL_BASE = "https://ep-twilight-snow-d88q9n3x.database.us-east-2.cloud.databricks.com/api/2.0/workspace/7474644702297592/rest/agent_knowledge_base"
    
    response = requests.post(
        f"{API_URL_BASE}/public/rpc/match_knowledge",
        headers={"Authorization": f"Bearer {sp_token}", "Content-Type": "application/json"},
        json={"query_embedding": question_embedding, "match_count": top_k}
    )
    return response.json()

In [0]:
results = vector_search("Why did order_reviews have duplicate records?")
print(results)